# Download MAS G1 Results

This notebook downloads MAS insurance company annual general business return workbooks for years 2022, 2023, and 2024 from the MAS insurance company returns page. Files are saved to `Data/MAS Form`, and a manifest CSV is written in the same folder.


In [3]:
#!/usr/bin/env python3
"""
Download MAS G1 insurance company return workbooks for selected years.

Default output folder:
    Data/MAS Form

Usage:
    python download_mas_g1_results.py
    python download_mas_g1_results.py --years 2022 2023 2024 --force
"""

from __future__ import annotations

import csv
import html
import os
import re
import time
from dataclasses import dataclass
from html.parser import HTMLParser
from pathlib import Path
from typing import Iterable
from urllib.error import HTTPError, URLError
from urllib.parse import unquote, urljoin, urlparse
from urllib.request import Request, urlopen


MAS_RETURNS_PAGE = (
    "https://www.mas.gov.sg/statistics/insurance-statistics/"
    "insurance-company-returns"
)
DEFAULT_YEARS = (2022, 2023, 2024)
EXCEL_EXTENSIONS = (".xlsx", ".xlsm", ".xls")
REQUEST_TIMEOUT_SECONDS = 60
MAX_RETRIES = 3


@dataclass(frozen=True)
class Candidate:
    url: str
    label: str
    year: int
    company: str
    referer: str
    institution_code: str


class LinkExtractor(HTMLParser):
    def __init__(self) -> None:
        super().__init__()
        self._current_href: str | None = None
        self._current_text: list[str] = []
        self.links: list[tuple[str, str]] = []

    def handle_starttag(self, tag: str, attrs: list[tuple[str, str | None]]) -> None:
        if tag.lower() != "a":
            return
        attrs_dict = {key.lower(): value for key, value in attrs}
        href = attrs_dict.get("href")
        if href:
            self._current_href = href
            self._current_text = []

    def handle_data(self, data: str) -> None:
        if self._current_href:
            self._current_text.append(data)

    def handle_endtag(self, tag: str) -> None:
        if tag.lower() == "a" and self._current_href:
            label = " ".join(" ".join(self._current_text).split())
            self.links.append((self._current_href, label))
            self._current_href = None
            self._current_text = []


def fetch_bytes(url: str, referer: str | None = None) -> bytes:
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0 Safari/537.36"
        ),
        "Accept": (
            "text/html,application/xhtml+xml,application/xml;q=0.9,"
            "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet,"
            "application/vnd.ms-excel,*/*;q=0.8"
        ),
        "Accept-Language": "en-US,en;q=0.9",
    }
    if referer:
        headers["Referer"] = referer
    request = Request(url, headers=headers)

    last_error: Exception | None = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with urlopen(request, timeout=REQUEST_TIMEOUT_SECONDS) as response:
                return response.read()
        except (HTTPError, URLError, TimeoutError) as exc:
            last_error = exc
            if attempt < MAX_RETRIES:
                time.sleep(2 * attempt)

    raise RuntimeError(f"Failed to fetch {url}: {last_error}") from last_error


def get_page_text(page_url: str, referer: str | None = None) -> str:
    content = fetch_bytes(page_url, referer=referer)
    text = content.decode("utf-8", errors="replace")

    unavailable_markers = (
        "Sorry, this service is currently unavailable",
        "<title>Maintenance</title>",
        "provide the ID number",
    )
    if any(marker.lower() in text.lower() for marker in unavailable_markers):
        raise RuntimeError(
            "MAS returned a temporary maintenance/service-unavailable page. "
            "Try again later, or run from a browser/network that can access MAS."
        )
    return text


def extract_links_from_text(text: str, page_url: str) -> list[tuple[str, str]]:

    parser = LinkExtractor()
    parser.feed(text)

    links = [(urljoin(page_url, html.unescape(href)), label) for href, label in parser.links]

    # Some MAS pages place file links in JSON/config blocks instead of anchors.
    raw_matches = re.findall(
        r"""(?:"|')(?P<url>(?:https?:)?//[^"']+\.(?:xlsx|xlsm|xls)(?:\?[^"']*)?|/[^"']+\.(?:xlsx|xlsm|xls)(?:\?[^"']*)?)(?:"|')""",
        text,
        flags=re.IGNORECASE,
    )
    for raw_url in raw_matches:
        links.append((urljoin(page_url, html.unescape(raw_url)), ""))

    deduped: dict[str, str] = {}
    for url, label in links:
        clean_url = url.split("#", 1)[0]
        deduped.setdefault(clean_url, label)
        if label and not deduped[clean_url]:
            deduped[clean_url] = label
    return list(deduped.items())


def extract_page_links(page_url: str, referer: str | None = None) -> list[tuple[str, str]]:
    return extract_links_from_text(get_page_text(page_url, referer=referer), page_url)


def is_excel_url(url: str) -> bool:
    path = unquote(urlparse(url).path).lower()
    return path.endswith(EXCEL_EXTENSIONS)


def find_year(text: str, years: Iterable[int]) -> int | None:
    for year in years:
        if re.search(rf"(?<!\d){year}(?!\d)", text):
            return year
    return None


def is_general_business_url(url: str) -> bool:
    normalized_path = unquote(urlparse(url).path).lower()
    return "/annual_g/" in normalized_path


def clean_company_name(value: str) -> str:
    text = html.unescape(value)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^A-Za-z0-9&+.-]+", " ", text)
    text = " ".join(text.split()).strip(" ._-")
    return text.upper() if text else "UNKNOWN_COMPANY"


def extract_company_title(page_text: str) -> str:
    for match in re.finditer(r"<h1[^>]*>(.*?)</h1>", page_text, flags=re.IGNORECASE | re.DOTALL):
        title = clean_company_name(match.group(1))
        if title and title not in {"UNKNOWN_COMPANY", "MAS SITE LOGO"}:
            return title

    title_match = re.search(
        r"<title[^>]*>(.*?)</title>", page_text, flags=re.IGNORECASE | re.DOTALL
    )
    return clean_company_name(title_match.group(1) if title_match else "UNKNOWN_COMPANY")


def institution_code_from_url(url: str) -> str:
    return Path(urlparse(url).path).name.upper()


def discover_institution_pages(page_url: str) -> list[str]:
    links = extract_page_links(page_url)
    institution_pages: dict[str, str] = {}
    pattern = re.compile(r"/statistics/insurance-statistics/insurance-company-returns/[a-z0-9]+$")
    for url, _label in links:
        parsed = urlparse(url)
        if pattern.search(parsed.path.lower()):
            institution_pages[url] = url
    return sorted(institution_pages)


def discover_candidates(page_url: str, years: Iterable[int]) -> list[Candidate]:
    selected_years = tuple(years)
    candidates: list[Candidate] = []

    for institution_url in discover_institution_pages(page_url):
        detail_text = get_page_text(institution_url, referer=page_url)
        company = extract_company_title(detail_text)
        institution_code = institution_code_from_url(institution_url)
        for url, label in extract_links_from_text(detail_text, institution_url):
            combined = f"{url} {label}"
            year = find_year(combined, selected_years)
            if not year or not is_excel_url(url) or not is_general_business_url(url):
                continue

            candidates.append(
                Candidate(
                    url=url,
                    label=label,
                    year=year,
                    company=company,
                    referer=institution_url,
                    institution_code=institution_code,
                )
            )

    return sorted(
        set(candidates),
        key=lambda item: (item.year, item.company, item.url),
    )


def safe_filename(candidate: Candidate) -> str:
    company = candidate.company
    year = candidate.year
    url = candidate.url
    suffix = Path(unquote(urlparse(url).path)).suffix.lower()
    if suffix not in EXCEL_EXTENSIONS:
        suffix = ".xlsx"
    company_part = re.sub(r"[^A-Za-z0-9&+.-]+", "_", company).strip("_")
    source_stem = Path(unquote(urlparse(url).path)).stem
    source_part = re.sub(r"[^A-Za-z0-9&+.-]+", "_", source_stem).strip("_")
    year_part = str(year)
    if source_part and source_part != year_part:
        year_part = f"{year_part}_{source_part}"
    return f"{candidate.institution_code}_{company_part}_{year_part}{suffix}"


def download_candidate(candidate: Candidate, output_dir: Path, force: bool) -> tuple[str, str]:
    output_path = output_dir / safe_filename(candidate)
    if output_path.exists() and not force:
        return ("skipped", str(output_path))

    data = fetch_bytes(candidate.url, referer=candidate.referer)
    if not data.startswith(b"PK") and not data.startswith(b"\xD0\xCF\x11\xE0"):
        raise RuntimeError(f"Downloaded file is not an Excel workbook: {candidate.url}")

    temporary_path = output_path.with_suffix(output_path.suffix + ".download")
    temporary_path.write_bytes(data)
    os.replace(temporary_path, output_path)
    return ("downloaded", str(output_path))


def write_manifest(output_dir: Path, rows: list[dict[str, str]]) -> Path:
    manifest_path = output_dir / "mas_g1_download_manifest.csv"
    fieldnames = [
        "status",
        "year",
        "institution_code",
        "company",
        "file",
        "url",
        "label",
        "error",
    ]
    with manifest_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return manifest_path


In [4]:
# Run this cell to download MAS G1 annual general business workbooks for 2022-2024.
# Files will be saved into Data/MAS Form. Set FORCE = False to skip files that already exist.

YEARS = (2022, 2023, 2024)
OUTPUT_DIR = Path("Data/MAS Form")
FORCE = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    candidates = discover_candidates(MAS_RETURNS_PAGE, YEARS)
except Exception as exc:
    raise RuntimeError(f"Discovery failed: {exc}") from exc

if not candidates:
    raise RuntimeError(
        "No MAS G1 Excel links were found for the requested years. "
        "The page structure may have changed, or MAS may be blocking the request."
    )

rows = []
for candidate in candidates:
    row = {
        "status": "",
        "year": str(candidate.year),
        "institution_code": candidate.institution_code,
        "company": candidate.company,
        "file": "",
        "url": candidate.url,
        "label": candidate.label,
        "error": "",
    }
    try:
        status, file_path = download_candidate(candidate, OUTPUT_DIR, FORCE)
        row["status"] = status
        row["file"] = file_path
        print(f"{status:10} {candidate.year} {candidate.company}: {file_path}")
    except Exception as exc:
        row["status"] = "failed"
        row["error"] = str(exc)
        print(f"failed     {candidate.year} {candidate.company}: {exc}")
    rows.append(row)

manifest_path = write_manifest(OUTPUT_DIR, rows)
print(f"\nManifest written to {manifest_path}")
print(f"Candidates found: {len(candidates)}")
print(f"Downloaded/skipped files: {sum(1 for r in rows if r['status'] in {'downloaded', 'skipped'})}")
print(f"Failed files: {sum(1 for r in rows if r['status'] == 'failed')}")


downloaded 2022 AIG ASIA PACIFIC INSURANCE PTE. LTD: Data/MAS Form/I870G_AIG_ASIA_PACIFIC_INSURANCE_PTE._LTD_2022.xlsx
downloaded 2022 ALLIANZ GLOBAL CORPORATE & SPECIALTY SE S BRANCH: Data/MAS Form/I874G_ALLIANZ_GLOBAL_CORPORATE_&_SPECIALTY_SE_S_BRANCH_2022.xlsx
downloaded 2022 ALLIED WORLD ASSURANCE COMPANY LTD S PORE BRANCH: Data/MAS Form/I868G_ALLIED_WORLD_ASSURANCE_COMPANY_LTD_S_PORE_BRANCH_2022.xlsx
downloaded 2022 ASPEN BERMUDA LIMITED SINGAPORE BRANCH: Data/MAS Form/R998G_ASPEN_BERMUDA_LIMITED_SINGAPORE_BRANCH_2022.xlsx
downloaded 2022 ASPEN INSURANCE UK LIMITED SINGAPORE BRANCH: Data/MAS Form/R980G_ASPEN_INSURANCE_UK_LIMITED_SINGAPORE_BRANCH_2022.xlsx
downloaded 2022 ASSURANCEFORENINGEN SKULD GJENSIDIG SINGAPORE BRANCH: Data/MAS Form/I881G_ASSURANCEFORENINGEN_SKULD_GJENSIDIG_SINGAPORE_BRANCH_2022.xlsx
downloaded 2022 ASTRO RE PTE. LTD: Data/MAS Form/RA16G_ASTRO_RE_PTE._LTD_2022.xlsx
downloaded 2022 ATRADIUS CREDITO Y CAUCION S.A. DE SEGUROS Y REASEGUROS: Data/MAS Form/I899G_AT

downloaded 2022 THE WEST OF ENGLAND SHIPOWNERS MUTUAL INSURANCE ASSOCIATION LUXEMBOURG SINGAPORE BRANCH: Data/MAS Form/I401G_THE_WEST_OF_ENGLAND_SHIPOWNERS_MUTUAL_INSURANCE_ASSOCIATION_LUXEMBOURG_SINGAPORE_BRANCH_2022.xlsx
downloaded 2022 TOKIO MARINE INSURANCE SINGAPORE LTD: Data/MAS Form/I728G_TOKIO_MARINE_INSURANCE_SINGAPORE_LTD_2022.xlsx
downloaded 2022 TOMONI RE PTE. LTD: Data/MAS Form/RA21G_TOMONI_RE_PTE._LTD_2022.xlsx
downloaded 2022 TRANSATLANTIC REINSURANCE COMPANY SINGAPORE BRANCH: Data/MAS Form/R988G_TRANSATLANTIC_REINSURANCE_COMPANY_SINGAPORE_BRANCH_2022.xlsx
downloaded 2022 TT CLUB MUTUAL INSURANCE LIMITED C O THOMAS MILLER SOUTH EAST ASIA PTE LTD: Data/MAS Form/I832G_TT_CLUB_MUTUAL_INSURANCE_LIMITED_C_O_THOMAS_MILLER_SOUTH_EAST_ASIA_PTE_LTD_2022.xlsx
downloaded 2022 UMIGAME RE PTE. LTD: Data/MAS Form/RA17G_UMIGAME_RE_PTE._LTD_2022.xlsx
downloaded 2022 UNITED OVERSEAS INSURANCE LTD: Data/MAS Form/I805G_UNITED_OVERSEAS_INSURANCE_LTD_2022.xlsx
downloaded 2022 XL INSURANCE CO

downloaded 2023 THE BRITANNIA STEAM SHIP INSURANCE ASSOCIATION EUROPE M.A. SINGAPORE BRANCH: Data/MAS Form/I406G_THE_BRITANNIA_STEAM_SHIP_INSURANCE_ASSOCIATION_EUROPE_M.A._SINGAPORE_BRANCH_2023.xlsx
downloaded 2023 THE JAPAN SHIP OWNERS MUTUAL P&I ASSN SINGAPORE BRANCH: Data/MAS Form/I883G_THE_JAPAN_SHIP_OWNERS_MUTUAL_P&I_ASSN_SINGAPORE_BRANCH_2023.xlsx
downloaded 2023 THE SHIPOWNERS MUTUAL P&I ASSN LUXEMBOURG C O SHIPOWNERS ASIA PTE LIMITED: Data/MAS Form/I863G_THE_SHIPOWNERS_MUTUAL_P&I_ASSN_LUXEMBOURG_C_O_SHIPOWNERS_ASIA_PTE_LIMITED_2023.xlsx
downloaded 2023 THE STANDARD CLUB ASIA LTD: Data/MAS Form/I830G_THE_STANDARD_CLUB_ASIA_LTD_2023.xlsx
downloaded 2023 THE SWEDISH CLUB SINGAPORE BRANCH: Data/MAS Form/I407G_THE_SWEDISH_CLUB_SINGAPORE_BRANCH_2023.xlsx
downloaded 2023 THE TOA REINSURANCE COMPANY LIMITED: Data/MAS Form/R955G_THE_TOA_REINSURANCE_COMPANY_LIMITED_2023.xlsx
downloaded 2023 THE UNITED KINGDOM MUTUAL STEAM SHIP ASSURANCE ASSOCIATION LIMITED SINGAPORE BRANCH FORMERLY KNOWN

downloaded 2024 PHOENIX 3 RE PTE. LTD: Data/MAS Form/RA26G_PHOENIX_3_RE_PTE._LTD_2024.xlsx
downloaded 2024 QBE INSURANCE SINGAPORE PTE. LTD: Data/MAS Form/I895G_QBE_INSURANCE_SINGAPORE_PTE._LTD_2024.xlsx
downloaded 2024 REARDON PTE LTD: Data/MAS Form/I866G_REARDON_PTE_LTD_2024.xlsx
downloaded 2024 RENAISSANCE REINSURANCE LTD. SINGAPORE BRANCH: Data/MAS Form/R989G_RENAISSANCE_REINSURANCE_LTD._SINGAPORE_BRANCH_2024.xlsx
downloaded 2024 SAMSUNG REINSURANCE PTE. LTD: Data/MAS Form/R987G_SAMSUNG_REINSURANCE_PTE._LTD_2024.xlsx
downloaded 2024 SEADRIF INSURANCE COMPANY PTE. LTD: Data/MAS Form/I405G_SEADRIF_INSURANCE_COMPANY_PTE._LTD_2024.xlsx
downloaded 2024 SINGAPORE REINSURANCE CORPORATION LTD: Data/MAS Form/R900G_SINGAPORE_REINSURANCE_CORPORATION_LTD_2024.xlsx
downloaded 2024 SOMPO INSURANCE SINGAPORE PTE. LTD: Data/MAS Form/I822G_SOMPO_INSURANCE_SINGAPORE_PTE._LTD_2024.xlsx
downloaded 2024 STARR INTERNATIONAL INSURANCE S PORE PTE. LTD: Data/MAS Form/I873G_STARR_INTERNATIONAL_INSURANCE_S_P